In [55]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict

project_root = Path.cwd()
if not (project_root / "runs").exists():
    project_root = project_root.parent

experiments = [
    "experiment_only_backprop_20260604_090527",
    "experiment_only_backprop_20260604_090543",
    "experiment_only_backprop_20260604_090556",
]

EXPERIMENT_DIRS = [project_root / "runs" / exp for exp in experiments]

# list of all subdirectories, instead of the logs
experiments = []
for exp_dir in EXPERIMENT_DIRS:
    for subdir in exp_dir.iterdir():
        if subdir.is_dir() and not subdir.name.startswith("logs"):
            experiments.append(str(subdir.relative_to(project_root)))

experiments

['runs\\experiment_only_backprop_20260604_090527\\bp_cnn_cifar100_proj2048_20260604_095348_seed1231',
 'runs\\experiment_only_backprop_20260604_090527\\bp_cnn_cifar10_proj2048_20260604_093058_seed1231',
 'runs\\experiment_only_backprop_20260604_090527\\bp_cnn_fashionmnist_proj2048_20260604_091754_seed1231',
 'runs\\experiment_only_backprop_20260604_090527\\bp_cnn_mnist_proj2048_20260604_090700_seed1231',
 'runs\\experiment_only_backprop_20260604_090543\\bp_cnn_cifar100_proj2048_20260604_110305_seed3845',
 'runs\\experiment_only_backprop_20260604_090543\\bp_cnn_cifar10_proj2048_20260604_104028_seed3845',
 'runs\\experiment_only_backprop_20260604_090543\\bp_cnn_fashionmnist_proj2048_20260604_102801_seed3845',
 'runs\\experiment_only_backprop_20260604_090543\\bp_cnn_mnist_proj2048_20260604_101646_seed3845',
 'runs\\experiment_only_backprop_20260604_090556\\bp_cnn_cifar100_proj2048_20260604_121230_seed92389',
 'runs\\experiment_only_backprop_20260604_090556\\bp_cnn_cifar10_proj2048_2026060

In [56]:
results = [str((project_root / "results" / e.split("\\")[2]).relative_to(project_root)) + ".json" for e in experiments]
results

['results\\bp_cnn_cifar100_proj2048_20260604_095348_seed1231.json',
 'results\\bp_cnn_cifar10_proj2048_20260604_093058_seed1231.json',
 'results\\bp_cnn_fashionmnist_proj2048_20260604_091754_seed1231.json',
 'results\\bp_cnn_mnist_proj2048_20260604_090700_seed1231.json',
 'results\\bp_cnn_cifar100_proj2048_20260604_110305_seed3845.json',
 'results\\bp_cnn_cifar10_proj2048_20260604_104028_seed3845.json',
 'results\\bp_cnn_fashionmnist_proj2048_20260604_102801_seed3845.json',
 'results\\bp_cnn_mnist_proj2048_20260604_101646_seed3845.json',
 'results\\bp_cnn_cifar100_proj2048_20260604_121230_seed92389.json',
 'results\\bp_cnn_cifar10_proj2048_20260604_114855_seed92389.json',
 'results\\bp_cnn_fashionmnist_proj2048_20260604_113652_seed92389.json',
 'results\\bp_cnn_mnist_proj2048_20260604_112607_seed92389.json']

In [57]:
from tbparse import SummaryReader
from typing import List

raw_dfs: List[pd.DataFrame] = [
    SummaryReader(exp_dir, extra_columns={"wall_time", "dir_name"}).scalars for exp_dir in EXPERIMENT_DIRS
]
df_all = pd.concat(raw_dfs, ignore_index=True)

df_all = df_all.assign(
    wall_clock=pd.to_datetime(df_all["wall_time"], unit="s", utc=True),
    architecture=df_all["dir_name"].str.extract(r"_(cnn|mlp)_")[0],
    dataset=df_all["dir_name"].str.extract(r"_(mnist|fashionmnist|cifar\d+)_")[0],
    train_method=df_all["dir_name"].str.extract(r"(bp|autodiff|dd\d+)_")[0],
    seed=df_all["dir_name"].str.extract(r"_seed(\d+)")[0].astype(int),
).drop(columns=["wall_time", "dir_name"])

arch_dfs: Dict[str, pd.DataFrame] = {
    arch: sub_df.drop(columns=["architecture"]).reset_index(drop=True)
    for arch, sub_df in df_all.groupby("architecture")
}

df_cnn = arch_dfs["cnn"]
df_cnn

,step,tag,value,wall_clock,dataset,train_method,seed
0,1,backprop/bp/acc/train,0.152156,2026-06-04 07:54:05.701327324+00:00,cifar100,bp,1231
1,2,backprop/bp/acc/train,0.275089,2026-06-04 07:54:12.121021986+00:00,cifar100,bp,1231
2,3,backprop/bp/acc/train,0.342422,2026-06-04 07:54:18.519614696+00:00,cifar100,bp,1231
3,4,backprop/bp/acc/train,0.392422,2026-06-04 07:54:24.950415850+00:00,cifar100,bp,1231
4,5,backprop/bp/acc/train,0.431756,2026-06-04 07:54:31.373302221+00:00,cifar100,bp,1231
...,...,...,...,...,...,...,...
9595,196,backprop/bp/loss/val,0.045304,2026-06-04 09:35:29.486784458+00:00,mnist,bp,92389
9596,197,backprop/bp/loss/val,0.045759,2026-06-04 09:35:32.319692612+00:00,mnist,bp,92389
9597,198,backprop/bp/loss/val,0.044901,2026-06-04 09:35:35.038608551+00:00,mnist,bp,92389
9598,199,backprop/bp/loss/val,0.045081,2026-06-04 09:35:37.832601547+00:00,mnist,bp,92389


In [58]:
import re


def parse_result(res_path: str):
    stem = Path(res_path).stem
    method = "bp"
    arch = re.search(r"_(cnn|mlp)_", stem).group(1)
    dataset = re.search(r"_(mnist|fashionmnist|cifar10|cifar100)_", stem).group(1)
    seed = int(re.search(r"_seed(\d+)$", stem).group(1))
    return method, arch, dataset, seed


In [59]:
import json

rows = []
for res in results:
    method, arch, dataset, seed = parse_result(res)
    data = json.loads((project_root / res).read_text())

    if method == "bp":
        rows.append(
            dict(
                method="bp",
                arch=arch,
                dataset=dataset,
                seed=seed,
                test_acc=data["bp"]["test_acc"],
            )
        )

raw = pd.DataFrame(rows)

raw.pivot_table(index=["arch", "method", "dataset"], columns="seed", values="test_acc").sort_index()

seed                       1231    3845    92389
arch method dataset                             
cnn  bp     cifar10       0.7296  0.7294  0.7343
            cifar100      0.4286  0.4294  0.4323
            fashionmnist  0.9095  0.9107  0.9114
            mnist         0.9886  0.9887  0.9899

In [60]:
agg = raw.groupby(["arch", "dataset"])["test_acc"].agg(mean="mean", std="std").reset_index()

DATASETS = ["mnist", "fashionmnist", "cifar10", "cifar100"]


def fmt(mean, std):
    return f"{mean:.4f}+-{std:.4f}"


def make_table(arch: str) -> pd.DataFrame:
    sub_agg = agg[agg["arch"] == arch]

    records = []
    for ds in DATASETS:
        s = sub_agg[sub_agg["dataset"] == ds]
        records.append({"dataset": ds, "BP": fmt(s["mean"].item(), s["std"].item())})
    return pd.DataFrame(records).set_index("dataset")


table_cnn = make_table("cnn")

print("\nCNN")
display(table_cnn)


CNN


,BP
dataset,
mnist,0.9891+-0.0007
fashionmnist,0.9105+-0.0010
cifar10,0.7311+-0.0028
cifar100,0.4301+-0.0019
